# Line Scoring Predictor

Trains an XGBoost model to predict P(score) for a given 7-player O-line.

**Features:** Aggregate per-player stats (mean, max, min, std) across the 7 players.  
**Label:** Did this O-point result in a goal before the next point start?  
**Output:** `models/lineup_xgb.pkl`

In [142]:
import psycopg2
import psycopg2.extras
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

DB_CONFIG = {
    'dbname': 'ufa_analytics',
    'user': os.getenv('DB_USER', 'postgres'),
    'password': '',
    'host': 'localhost',
    'port': 5432
}

def get_connection():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.cursor_factory = psycopg2.extras.RealDictCursor
    return conn

print('Imports OK')

Imports OK


## Step 1: Per-player stats

In [143]:
conn = get_connection()
cur = conn.cursor()

cur.execute("""
    SELECT
        thrower,
        COUNT(*) as total_throws,
        SUM(CASE WHEN event_type IN (18, 19) THEN 1 ELSE 0 END) as completions,
        SUM(CASE WHEN event_type = 19 THEN 1 ELSE 0 END) as goals,
        AVG(CASE
            WHEN event_type IN (18, 19) AND receiver_x IS NOT NULL AND receiver_y IS NOT NULL
            THEN SQRT(POWER(receiver_x - thrower_x, 2) + POWER(receiver_y - thrower_y, 2))
        END) as avg_throw_dist,
        AVG(CASE
            WHEN event_type IN (18, 19) AND receiver_y IS NOT NULL
            THEN receiver_y - thrower_y
        END) as avg_throw_depth,
        SUM(CASE
            WHEN event_type IN (18, 19) AND receiver_y IS NOT NULL
                AND ABS(receiver_y - thrower_y) > 30
            THEN 1 ELSE 0
        END)::float / NULLIF(SUM(CASE WHEN event_type IN (18, 19) THEN 1 ELSE 0 END), 0) as huck_rate,
        AVG(CASE
            WHEN event_type IN (18, 19) AND receiver_x IS NOT NULL
            THEN ABS(receiver_x - thrower_x)
        END) as avg_lateral_dist,
        AVG(CASE
            WHEN thrower_x IS NOT NULL
            THEN ABS(thrower_x)
        END) as avg_dist_from_center
    FROM events
    WHERE thrower IS NOT NULL AND event_type IN (18, 19, 20, 22)
    GROUP BY thrower
    HAVING COUNT(*) >= 50
""")

player_stats = {}
for row in cur.fetchall():
    p = row['thrower']
    total = row['total_throws']
    completions = row['completions']
    player_stats[p] = {
        'completion_pct': round(100 * completions / total, 1) if total > 0 else 0,
        'goal_pct':       round(100 * row['goals'] / total, 1) if total > 0 else 0,
        'avg_throw_dist': round(float(row['avg_throw_dist'] or 0), 1),
        'avg_throw_depth':round(float(row['avg_throw_depth'] or 0), 1),
        'huck_rate':      round(100 * float(row['huck_rate'] or 0), 1),
        'avg_lateral_dist':     round(float(row['avg_lateral_dist'] or 0), 1),
        'avg_dist_from_center': round(float(row['avg_dist_from_center'] or 0), 1),
    }

cur.close()
conn.close()

print(f'Players with stats: {len(player_stats)}')

STAT_COLS = ['completion_pct', 'goal_pct', 'avg_throw_dist', 'avg_throw_depth',
             'huck_rate', 'avg_lateral_dist', 'avg_dist_from_center']

league_avg = {col: np.mean([player_stats[p][col] for p in player_stats]) for col in STAT_COLS}
print('\nLeague averages:')
for k, v in league_avg.items():
    print(f'  {k}: {v:.2f}')

Players with stats: 1060

League averages:
  completion_pct: 92.26
  goal_pct: 7.44
  avg_throw_dist: 15.47
  avg_throw_depth: 0.07
  huck_rate: 4.34
  avg_lateral_dist: 9.31
  avg_dist_from_center: 11.09


In [144]:
## Step 1b: Per-player receiver stats — merge into player_stats

conn = get_connection()
cur = conn.cursor()

cur.execute("""
    SELECT
        receiver,
        COUNT(*) as total_targets,
        SUM(CASE WHEN event_type IN (18, 19) THEN 1 ELSE 0 END) as catches,
        AVG(CASE
            WHEN event_type IN (18, 19) AND receiver_x IS NOT NULL
            THEN SQRT(POWER(receiver_x - thrower_x, 2) + POWER(receiver_y - thrower_y, 2))
        END) as avg_recv_dist,
        AVG(CASE
            WHEN event_type IN (18, 19) AND receiver_y IS NOT NULL
            THEN receiver_y - thrower_y
        END) as avg_recv_depth
    FROM events
    WHERE receiver IS NOT NULL AND event_type IN (18, 19, 20)
    GROUP BY receiver
    HAVING COUNT(*) >= 30
""")

recv_stats = {}
for row in cur.fetchall():
    p = row['receiver']
    total = row['total_targets']
    catches = row['catches']
    recv_stats[p] = {
        'catch_rate':      round(100 * catches / total, 1) if total > 0 else 0,
        'avg_recv_dist':   round(float(row['avg_recv_dist'] or 0), 1),
        'avg_recv_depth':  round(float(row['avg_recv_depth'] or 0), 1),
    }

cur.close()
conn.close()

print(f'Players with receiver stats: {len(recv_stats)}')

# Receiver league averages (used as fallback for players without enough receiving data)
RECV_STAT_COLS = ['catch_rate', 'avg_recv_dist', 'avg_recv_depth']
recv_league_avg = {
    col: np.mean([recv_stats[p][col] for p in recv_stats])
    for col in RECV_STAT_COLS
}
print('Receiver league averages:')
for k, v in recv_league_avg.items():
    print(f'  {k}: {v:.2f}')

# Merge receiver stats into player_stats; use league avg for missing players
for p in player_stats:
    r = recv_stats.get(p, recv_league_avg)
    for col in RECV_STAT_COLS:
        player_stats[p][col] = r[col]

# Also add receiver-only players to player_stats using recv league avg for thrower fields
# (skip — we only want players who appear in lineup_players, thrower stats are the anchor)

# Update STAT_COLS to include receiver features
STAT_COLS = STAT_COLS + RECV_STAT_COLS
league_avg.update(recv_league_avg)

print(f'\nTotal stat columns: {len(STAT_COLS)}')
print('STAT_COLS:', STAT_COLS)

Players with receiver stats: 1254
Receiver league averages:
  catch_rate: 98.43
  avg_recv_dist: 16.62
  avg_recv_depth: 0.09

Total stat columns: 10
STAT_COLS: ['completion_pct', 'goal_pct', 'avg_throw_dist', 'avg_throw_depth', 'huck_rate', 'avg_lateral_dist', 'avg_dist_from_center', 'catch_rate', 'avg_recv_dist', 'avg_recv_depth']


In [145]:
## Step 1c: Per-player O-line hold rate — merge into player_stats

conn = get_connection()
cur = conn.cursor()

cur.execute("""
    WITH o_starts AS (
        SELECT e.event_id, e.game_id, e.event_number, e.team
        FROM events e WHERE e.event_type = 2
    ),
    outcomes AS (
        SELECT os.event_id,
            COALESCE(BOOL_OR(e2.event_type = 19 AND e2.team = os.team), FALSE) AS scored
        FROM o_starts os
        LEFT JOIN events e2 ON e2.game_id = os.game_id
            AND e2.event_number > os.event_number
            AND e2.event_number < COALESCE(
                (SELECT MIN(e3.event_number) FROM events e3
                 WHERE e3.game_id = os.game_id AND e3.event_type = 1
                   AND e3.event_number > os.event_number),
                9999999
            )
        GROUP BY os.event_id
    )
    SELECT
        lp.player_id,
        COUNT(*) AS o_possessions,
        AVG(CASE WHEN oc.scored THEN 1.0 ELSE 0.0 END) AS o_hold_rate
    FROM line_players lp
    JOIN outcomes oc ON oc.event_id = lp.event_id
    WHERE lp.line_type = 'O'
    GROUP BY lp.player_id
    HAVING COUNT(*) >= 20
""")

hold_stats = {
    row['player_id']: {
        'rate':        float(row['o_hold_rate']),
        'possessions': int(row['o_possessions']),
    }
    for row in cur.fetchall()
}

cur.close()
conn.close()

print(f'Players with O-line hold rate: {len(hold_stats)}')

# League average hold rate
hold_league_avg = float(np.mean([v['rate'] for v in hold_stats.values()]))
print(f'League avg O-line hold rate: {hold_league_avg:.3f}')

# Merge into player_stats; use league avg for players below threshold
for p in player_stats:
    player_stats[p]['o_hold_rate'] = hold_stats[p]['rate'] if p in hold_stats else hold_league_avg

# Update STAT_COLS and league_avg
STAT_COLS = STAT_COLS + ['o_hold_rate']
league_avg['o_hold_rate'] = hold_league_avg

print(f'\nTotal stat columns: {len(STAT_COLS)}')
print('STAT_COLS:', STAT_COLS)

Players with O-line hold rate: 1025
League avg O-line hold rate: 0.603

Total stat columns: 11
STAT_COLS: ['completion_pct', 'goal_pct', 'avg_throw_dist', 'avg_throw_depth', 'huck_rate', 'avg_lateral_dist', 'avg_dist_from_center', 'catch_rate', 'avg_recv_dist', 'avg_recv_depth', 'o_hold_rate']


In [146]:
## Step 1d: Pairwise line familiarity — shared O-line co-occurrence counts

conn = get_connection()
cur = conn.cursor()

cur.execute("""
    SELECT lp1.player_id AS p1, lp2.player_id AS p2, COUNT(*) AS shared_lines
    FROM line_players lp1
    JOIN line_players lp2
        ON lp1.event_id = lp2.event_id
        AND lp1.line_type = 'O'
        AND lp2.line_type = 'O'
        AND lp1.player_id < lp2.player_id
    GROUP BY lp1.player_id, lp2.player_id
""")

pair_familiarity = {(row['p1'], row['p2']): int(row['shared_lines']) for row in cur.fetchall()}

cur.close()
conn.close()

print(f'Unique player pairs with shared O-line history: {len(pair_familiarity)}')
counts = list(pair_familiarity.values())
print(f'Shared lines per pair — mean: {np.mean(counts):.1f}, median: {np.median(counts):.1f}, max: {max(counts)}')

Unique player pairs with shared O-line history: 22608
Shared lines per pair — mean: 30.5, median: 6.0, max: 951


## Step 2: Fetch O-point starts with lineups and outcomes

In [147]:
conn = get_connection()
cur = conn.cursor()

cur.execute("""
    WITH o_starts AS (
        SELECT e.event_id, e.game_id, e.event_number, e.team
        FROM events e
        WHERE e.event_type = 2
    ),
    -- Paired D-start: event_type=1 always fires at event_number - 1 (pulling team before receiving team)
    d_starts AS (
        SELECT os.event_id AS o_event_id,
               e2.event_id AS d_event_id
        FROM o_starts os
        JOIN events e2 ON e2.game_id = os.game_id
            AND e2.event_type = 1
            AND e2.event_number = os.event_number - 1
    ),
    outcomes AS (
        SELECT os.event_id,
            COALESCE(BOOL_OR(e2.event_type = 19 AND e2.team = os.team), FALSE) AS scored
        FROM o_starts os
        LEFT JOIN events e2 ON e2.game_id = os.game_id
            AND e2.event_number > os.event_number
            AND e2.event_number < COALESCE(
                (SELECT MIN(e3.event_number) FROM events e3
                 WHERE e3.game_id = os.game_id AND e3.event_type = 1
                   AND e3.event_number > os.event_number),
                9999999
            )
        GROUP BY os.event_id
    ),
    goal_context AS (
        SELECT os.event_id, os.team,
            COUNT(CASE WHEN g.defending_team != os.team AND g.event_number < os.event_number THEN 1 END) AS o_score,
            COUNT(CASE WHEN g.defending_team  = os.team AND g.event_number < os.event_number THEN 1 END) AS d_score
        FROM o_starts os
        LEFT JOIN (
            SELECT game_id, event_number, team AS defending_team
            FROM events WHERE event_type = 19
        ) g ON g.game_id = os.game_id
        GROUP BY os.event_id, os.team
    ),
    -- Each quarter-boundary event fires TWICE (once per team), so use
    -- COUNT(DISTINCT event_type) to get 0-6 distinct boundaries → quarters 1-6.
    -- Types: 28=Q1-end, 29=Q2-end, 30=Q3-end, 31=Q4-end, 32=OT1-end, 33=OT2-end.
    quarter_context AS (
        SELECT os.event_id,
            COUNT(DISTINCT q.event_type) + 1 AS quarter
        FROM o_starts os
        LEFT JOIN (
            SELECT game_id, event_number, event_type
            FROM events WHERE event_type IN (28, 29, 30, 31, 32, 33)
        ) q ON q.game_id = os.game_id AND q.event_number < os.event_number
        GROUP BY os.event_id, os.event_number
    ),
    context AS (
        SELECT gc.event_id, gc.o_score, gc.d_score, qc.quarter
        FROM goal_context gc
        JOIN quarter_context qc ON qc.event_id = gc.event_id
    )
    SELECT o.event_id, o.team, oc.scored,
           ARRAY_AGG(DISTINCT op.player_id) FILTER (WHERE op.player_id IS NOT NULL) AS o_lineup,
           ARRAY_AGG(DISTINCT dp.player_id) FILTER (WHERE dp.player_id IS NOT NULL) AS d_lineup,
           ctx.o_score, ctx.d_score, ctx.quarter
    FROM o_starts o
    JOIN outcomes oc ON o.event_id = oc.event_id
    JOIN context ctx ON ctx.event_id = o.event_id
    JOIN line_players op ON op.event_id = o.event_id AND op.line_type = 'O'
    LEFT JOIN d_starts ds ON ds.o_event_id = o.event_id
    LEFT JOIN line_players dp ON dp.event_id = ds.d_event_id AND dp.line_type = 'D'
    GROUP BY o.event_id, o.team, oc.scored, ctx.o_score, ctx.d_score, ctx.quarter
    HAVING COUNT(DISTINCT op.player_id) = 7
""")

rows = cur.fetchall()
cur.close()
conn.close()

print(f'O-point possessions with full O-lineups: {len(rows)}')
scored = sum(1 for r in rows if r['scored'])
print(f'Scoring rate: {100*scored/len(rows):.1f}%')

import statistics
diffs = [int(r['o_score']) - int(r['d_score']) for r in rows]
quarters = [int(r['quarter']) for r in rows]
print(f'\nScore diff: mean={statistics.mean(diffs):.2f}, range=[{min(diffs)}, {max(diffs)}]')
print(f'Quarter distribution: { {q: quarters.count(q) for q in sorted(set(quarters))} }')

O-point possessions with full O-lineups: 32848
Scoring rate: 64.1%

Score diff: mean=1.02, range=[-23, 25]
Quarter distribution: {1: 8172, 2: 8155, 3: 8195, 4: 8128, 5: 185, 6: 13}


## Step 3: Feature engineering

In [148]:
from itertools import combinations

def lineup_to_features(o_lineup, d_lineup):
    """
    Aggregate O-line and D-line player stats into a feature vector.
    D-line falls back to league averages if players are missing from stats.
    At inference time, pass d_lineup=[] to use full league averages for D.
    """
    o_stats = [[player_stats[pid][col] for col in STAT_COLS]
               for pid in o_lineup if pid in player_stats]
    d_stats = [[player_stats[pid][col] for col in STAT_COLS]
               for pid in (d_lineup or []) if pid in player_stats]

    if not o_stats:
        return None, 0, 0

    o_arr = np.array(o_stats)

    if d_stats:
        d_arr = np.array(d_stats)
    else:
        d_arr = np.array([[league_avg[col] for col in STAT_COLS]])

    feats = []
    for arr in [o_arr, d_arr]:
        for col_vals in arr.T:
            feats += [float(col_vals.mean()), float(col_vals.max()),
                      float(col_vals.min()), float(col_vals.std())]

    feats.append(float(np.sum(o_arr[:, STAT_COLS.index('completion_pct')] > league_avg['completion_pct'])))
    feats.append(float(np.sum(o_arr[:, STAT_COLS.index('huck_rate')] > league_avg['huck_rate'])))
    feats.append(float(len(o_stats)))
    feats.append(float(len(d_stats)))

    # Pairwise line familiarity: 21 pairs in a 7-player lineup
    pair_counts = []
    for p1, p2 in combinations(sorted(o_lineup), 2):
        key = (p1, p2) if p1 < p2 else (p2, p1)
        pair_counts.append(pair_familiarity.get(key, 0))

    if pair_counts:
        feats += [
            float(np.mean(pair_counts)),               # avg shared O-lines per pair
            float(np.min(pair_counts)),                # weakest link
            float(np.max(pair_counts)),                # strongest pair
            float(sum(1 for c in pair_counts if c > 0)),  # n pairs with any history
        ]
    else:
        feats += [0.0, 0.0, 0.0, 0.0]

    return feats, len(o_stats), len(d_stats)

# Build feature names (game context appended separately in the training loop)
FEATURE_NAMES = []
for prefix in ['o', 'd']:
    for col in STAT_COLS:
        for agg in ['mean', 'max', 'min', 'std']:
            FEATURE_NAMES.append(f'{prefix}_{col}_{agg}')
FEATURE_NAMES += ['n_above_avg_completion', 'n_above_avg_huck', 'n_o_players_with_stats', 'n_d_players_with_stats']
FEATURE_NAMES += ['pair_familiarity_mean', 'pair_familiarity_min', 'pair_familiarity_max', 'n_pairs_with_history']
FEATURE_NAMES += ['score_diff', 'total_score', 'quarter']

print(f'Feature count: {len(FEATURE_NAMES)}')

Feature count: 99


In [149]:
X_list, y_list = [], []
skipped = 0

for row in rows:
    feats, n_o, n_d = lineup_to_features(row['o_lineup'], row['d_lineup'])
    if feats is None or n_o < 3:
        skipped += 1
        continue

    # Append game context features
    o_score = int(row['o_score'])
    d_score = int(row['d_score'])
    feats += [
        float(o_score - d_score),           # score_diff (negative = trailing)
        float(o_score + d_score),           # total_score (game pacing / fatigue)
        float(min(int(row['quarter']), 6)), # quarter capped at 6 (Q1-Q4 + OT1 + OT2)
    ]

    X_list.append(feats)
    y_list.append(1 if row['scored'] else 0)

X = np.array(X_list)
y = np.array(y_list)

print(f'Training samples: {len(X)}')
print(f'Skipped (too few O players with stats): {skipped}')
print(f'Class balance: {y.mean():.3f} scoring rate')
print(f'Features per sample: {X.shape[1]}')

Training samples: 32787
Skipped (too few O players with stats): 61
Class balance: 0.642 scoring rate
Features per sample: 99


## Step 4: Train XGBoost

In [150]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=333, stratify=y
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=333
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_prob)
acc = accuracy_score(y_test, y_pred)
print(f'\nTest AUC:      {auc:.4f}')
print(f'Test Accuracy: {acc:.4f}')

[0]	validation_0-logloss:0.64993
[50]	validation_0-logloss:0.62184


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:50:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[100]	validation_0-logloss:0.61956
[150]	validation_0-logloss:0.61958
[200]	validation_0-logloss:0.62020
[250]	validation_0-logloss:0.62037
[299]	validation_0-logloss:0.62099

Test AUC:      0.6466
Test Accuracy: 0.6670


In [151]:
## Step 4b: LightGBM + Optuna hyperparameter search

import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Hold out a validation slice from X_train for Optuna trials (keep X_test unseen)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=333, stratify=y_train
)

def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'verbosity':         -1,
        'n_estimators':      1000,
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    clf = lgb.LGBMClassifier(**params)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)],
    )
    return roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Best trial AUC (val): {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

  0%|          | 0/50 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/

Best trial AUC (val): 0.6605
Best params:
  num_leaves: 28
  learning_rate: 0.020140420125531952
  min_child_samples: 24
  feature_fraction: 0.9460617756982411
  bagging_fraction: 0.5485083289805772
  bagging_freq: 1
  reg_alpha: 3.1266483863764923e-07
  reg_lambda: 2.0619087885429512


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [152]:
## Step 4c: Train final LightGBM on full X_train + compare with XGBoost

lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    verbosity=-1,
    n_estimators=1000,
    **study.best_params,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=50)],
)

lgb_preds = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_preds)
lgb_acc = accuracy_score(y_test, (lgb_preds >= 0.5).astype(int))

print(f'LightGBM  → AUC: {lgb_auc:.4f} | Accuracy: {lgb_acc:.4f}')
print(f'XGBoost   → AUC: {auc:.4f}  | Accuracy: {acc:.4f}')

if lgb_auc >= auc:
    best_model = lgb_model
    best_auc, best_acc = lgb_auc, lgb_acc
    model_type = 'lightgbm'
    print(f'\n→ LightGBM wins  (+{lgb_auc - auc:.4f} AUC)')
else:
    best_model = model   # XGBoost from cell k11
    best_auc, best_acc = auc, acc
    model_type = 'xgboost'
    print(f'\n→ XGBoost holds  (+{auc - lgb_auc:.4f} AUC)')

[50]	valid_0's auc: 0.641692
[100]	valid_0's auc: 0.644735
[150]	valid_0's auc: 0.64673
[200]	valid_0's auc: 0.647046
LightGBM  → AUC: 0.6477 | Accuracy: 0.6697
XGBoost   → AUC: 0.6466  | Accuracy: 0.6670

→ LightGBM wins  (+0.0011 AUC)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Step 5: Feature importance

In [153]:
importances = best_model.feature_importances_
feat_imp = sorted(zip(FEATURE_NAMES, importances), key=lambda x: x[1], reverse=True)
print(f'Top 15 features ({model_type}):')
for name, imp in feat_imp[:15]:
    print(f'  {name:<40} {imp:.4f}')

Top 15 features (lightgbm):
  o_o_hold_rate_min                        185.0000
  pair_familiarity_min                     153.0000
  d_goal_pct_mean                          140.0000
  total_score                              134.0000
  o_o_hold_rate_max                        131.0000
  o_o_hold_rate_mean                       93.0000
  n_d_players_with_stats                   92.0000
  d_completion_pct_mean                    88.0000
  d_avg_recv_depth_std                     82.0000
  o_avg_recv_dist_mean                     79.0000
  d_o_hold_rate_max                        78.0000
  d_o_hold_rate_mean                       76.0000
  d_avg_lateral_dist_std                   74.0000
  pair_familiarity_mean                    71.0000
  d_avg_recv_dist_mean                     70.0000


## Step 6: Save model

In [154]:
save = {
    'model':            best_model,
    'model_type':       model_type,
    'player_stats':     player_stats,       # thrower + receiver + hold rate stats
    'pair_familiarity': pair_familiarity,   # (p1, p2) -> shared O-line count
    'feature_names':    FEATURE_NAMES,
    'stat_cols':        STAT_COLS,
    'league_avg':       league_avg,
    'metrics':          {'auc': round(best_auc, 4), 'accuracy': round(best_acc, 4)},
}

out_path = Path('models/lineup_xgb.pkl')
out_path.parent.mkdir(exist_ok=True)
joblib.dump(save, out_path)
print(f'Saved to {out_path}  ({model_type})')
print(f'   AUC: {best_auc:.4f} | Accuracy: {best_acc:.4f}')
print(f'   Players in stats dict: {len(player_stats)}')
print(f'   Pair familiarity entries: {len(pair_familiarity)}')
print(f'   Stat columns ({len(STAT_COLS)}): {STAT_COLS}')
print(f'   Features: {len(FEATURE_NAMES)}')

Saved to models/lineup_xgb.pkl  (lightgbm)
   AUC: 0.6477 | Accuracy: 0.6697
   Players in stats dict: 1060
   Pair familiarity entries: 22608
   Stat columns (11): ['completion_pct', 'goal_pct', 'avg_throw_dist', 'avg_throw_depth', 'huck_rate', 'avg_lateral_dist', 'avg_dist_from_center', 'catch_rate', 'avg_recv_dist', 'avg_recv_depth', 'o_hold_rate']
   Features: 99
